In [2]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

project_root = Path(
    r"D:\PROGRAMMING\NetGuard AI\netguard_ai"
)

train_df = pd.read_csv(
    project_root / "data/raw/UNSW_NB15_training-set.csv"
)

feature_columns = [
    column
    for column in train_df.columns
    if column not in [
        "id", "label", "attack_cat", "is_ftp_login"
    ]
]

attack_data = (
    train_df.loc[
        train_df["label"] == 1,
        feature_columns + ["attack_cat"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Attack dataset shape:", attack_data.shape)
print("\nAttack category counts:")
print(attack_data["attack_cat"].value_counts())

Attack dataset shape: (55850, 42)

Attack category counts:
attack_cat
Exploits          19844
Fuzzers           16150
Reconnaissance     7522
Generic            4181
DoS                3806
Analysis           1594
Backdoor           1535
Shellcode          1091
Worms               127
Name: count, dtype: int64


In [3]:
X_attack = attack_data[feature_columns].copy()
y_attack = attack_data["attack_cat"].copy()

attack_groups = pd.util.hash_pandas_object(
    X_attack,
    index=False
)

print("Input features:", X_attack.shape[1])
print("Attack categories:", y_attack.nunique())
print("Unique feature patterns:", attack_groups.nunique())

Input features: 41
Attack categories: 9
Unique feature patterns: 49379


In [4]:
attack_splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

attack_train_indices, attack_val_indices = next(
    attack_splitter.split(
        X_attack,
        y_attack,
        groups=attack_groups
    )
)

X_attack_train = X_attack.iloc[attack_train_indices].copy()
X_attack_validation = X_attack.iloc[attack_val_indices].copy()

y_attack_train = y_attack.iloc[attack_train_indices].copy()
y_attack_validation = y_attack.iloc[attack_val_indices].copy()

overlap = set(
    attack_groups.iloc[attack_train_indices]
).intersection(
    attack_groups.iloc[attack_val_indices]
)

print("Training shape:", X_attack_train.shape)
print("Validation shape:", X_attack_validation.shape)
print("Feature-group overlap:", len(overlap))

category_counts = pd.DataFrame({
    "training": y_attack_train.value_counts(),
    "validation": y_attack_validation.value_counts()
}).fillna(0).astype(int)

print("\nCategory counts:")
print(category_counts.to_string())

assert len(overlap) == 0
assert set(y_attack_train) == set(y_attack)
assert set(y_attack_validation) == set(y_attack)

print("\nSplit checks passed")

Training shape: (44680, 41)
Validation shape: (11170, 41)
Feature-group overlap: 0

Category counts:
                training  validation
attack_cat                          
Exploits           15876        3968
Fuzzers            12920        3230
Reconnaissance      6017        1505
Generic             3344         837
DoS                 3045         761
Analysis            1276         318
Backdoor            1228         307
Shellcode            873         218
Worms                101          26

Split checks passed


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

numeric_columns = (
    X_attack_train.select_dtypes(include="number")
    .columns.tolist()
)

categorical_columns = (
    X_attack_train.select_dtypes(exclude="number")
    .columns.tolist()
)

attack_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            SimpleImputer(strategy="median"),
            numeric_columns
        ),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True
                ))
            ]),
            categorical_columns
        )
    ]
)

attack_pipeline = Pipeline([
    ("preprocessor", attack_preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight=None,
        random_state=42,
        n_jobs=2
    ))
])

In [6]:
from time import perf_counter

start_time = perf_counter()

attack_pipeline.fit(
    X_attack_train,
    y_attack_train
)

print(
    f"Training time: {perf_counter() - start_time:.2f} seconds"
)

attack_train_predictions = attack_pipeline.predict(
    X_attack_train
)

attack_validation_predictions = attack_pipeline.predict(
    X_attack_validation
)

Training time: 21.43 seconds


In [7]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

attack_classes = (
    attack_pipeline.named_steps["classifier"].classes_
)

print(
    classification_report(
        y_attack_validation,
        attack_validation_predictions,
        labels=attack_classes,
        digits=4,
        zero_division=0
    )
)

print(
    "Training Macro F1:",
    round(f1_score(
        y_attack_train,
        attack_train_predictions,
        average="macro",
        zero_division=0
    ), 4)
)

print(
    "Validation Macro F1:",
    round(f1_score(
        y_attack_validation,
        attack_validation_predictions,
        average="macro",
        zero_division=0
    ), 4)
)

print(
    "Validation accuracy:",
    round(accuracy_score(
        y_attack_validation,
        attack_validation_predictions
    ), 4)
)

attack_confusion = confusion_matrix(
    y_attack_validation,
    attack_validation_predictions,
    labels=attack_classes
)

confusion_table = pd.DataFrame(
    attack_confusion,
    index=attack_classes,
    columns=attack_classes
)

print("\nConfusion matrix: rows = actual, columns = predicted")
print(confusion_table.to_string())

                precision    recall  f1-score   support

      Analysis     0.2683    0.3459    0.3022       318
      Backdoor     0.2427    0.2964    0.2669       307
           DoS     0.5270    0.1537    0.2380       761
      Exploits     0.8081    0.8758    0.8406      3968
       Fuzzers     0.8287    0.9211    0.8724      3230
       Generic     0.9970    0.8041    0.8902       837
Reconnaissance     0.8081    0.7389    0.7720      1505
     Shellcode     0.6182    0.6239    0.6210       218
         Worms     1.0000    0.0769    0.1429        26

      accuracy                         0.7781     11170
     macro avg     0.6776    0.5374    0.5496     11170
  weighted avg     0.7749    0.7781    0.7662     11170

Training Macro F1: 0.6468
Validation Macro F1: 0.5496
Validation accuracy: 0.7781

Confusion matrix: rows = actual, columns = predicted
                Analysis  Backdoor  DoS  Exploits  Fuzzers  Generic  Reconnaissance  Shellcode  Worms
Analysis             110       

In [8]:
from sklearn.base import clone

weighted_attack_pipeline = clone(attack_pipeline)

weighted_attack_pipeline.set_params(
    classifier__class_weight="balanced"
)

start_time = perf_counter()

weighted_attack_pipeline.fit(
    X_attack_train,
    y_attack_train
)

print(
    f"Training time: {perf_counter() - start_time:.2f} seconds"
)

weighted_train_predictions = weighted_attack_pipeline.predict(
    X_attack_train
)

weighted_validation_predictions = weighted_attack_pipeline.predict(
    X_attack_validation
)

print(
    classification_report(
        y_attack_validation,
        weighted_validation_predictions,
        labels=attack_classes,
        digits=4,
        zero_division=0
    )
)

print(
    "Training Macro F1:",
    round(f1_score(
        y_attack_train,
        weighted_train_predictions,
        average="macro",
        zero_division=0
    ), 4)
)

print(
    "Validation Macro F1:",
    round(f1_score(
        y_attack_validation,
        weighted_validation_predictions,
        average="macro",
        zero_division=0
    ), 4)
)

Training time: 17.15 seconds
                precision    recall  f1-score   support

      Analysis     0.2288    0.6101    0.3328       318
      Backdoor     0.2200    0.6156    0.3242       307
           DoS     0.4746    0.3443    0.3991       761
      Exploits     0.9152    0.7130    0.8015      3968
       Fuzzers     0.9444    0.8780    0.9100      3230
       Generic     0.9576    0.8363    0.8929       837
Reconnaissance     0.7735    0.8100    0.7913      1505
     Shellcode     0.4372    0.9266    0.5941       218
         Worms     0.3750    0.6923    0.4865        26

      accuracy                         0.7564     11170
     macro avg     0.5918    0.7140    0.6147     11170
  weighted avg     0.8285    0.7564    0.7797     11170

Training Macro F1: 0.7022
Validation Macro F1: 0.6147


In [9]:
test_df = pd.read_csv(
    project_root / "data/raw/UNSW_NB15_testing-set.csv"
)

# Evaluate category classification on actual attack records
test_attacks = test_df.loc[test_df["label"] == 1].copy()

X_attack_test = test_attacks[feature_columns]
y_attack_test = test_attacks["attack_cat"]

attack_test_predictions = weighted_attack_pipeline.predict(
    X_attack_test
)

print("Official attack-only test records:", len(X_attack_test))

print(
    classification_report(
        y_attack_test,
        attack_test_predictions,
        labels=attack_classes,
        digits=4,
        zero_division=0
    )
)

print(
    "Official test Macro F1:",
    round(f1_score(
        y_attack_test,
        attack_test_predictions,
        labels=attack_classes,
        average="macro",
        zero_division=0
    ), 4)
)

Official attack-only test records: 45332
                precision    recall  f1-score   support

      Analysis     0.0891    0.3323    0.1406       677
      Backdoor     0.0472    0.4597    0.0856       583
           DoS     0.4631    0.1460    0.2220      4089
      Exploits     0.8732    0.5714    0.6908     11132
       Fuzzers     0.8329    0.7095    0.7663      6062
       Generic     0.9978    0.9702    0.9838     18871
Reconnaissance     0.8406    0.8464    0.8435      3496
     Shellcode     0.2444    0.9312    0.3872       378
         Worms     0.3882    0.7500    0.5116        44

      accuracy                         0.7369     45332
     macro avg     0.5307    0.6352    0.5146     45332
  weighted avg     0.8521    0.7369    0.7736     45332

Official test Macro F1: 0.5146


In [10]:
development_hashes = pd.util.hash_pandas_object(
    X_attack,
    index=False
)

test_attack_hashes = pd.util.hash_pandas_object(
    X_attack_test,
    index=False
)

novel_attack_mask = ~test_attack_hashes.isin(
    set(development_hashes)
)

print(
    "Novel attack-pattern test records:",
    int(novel_attack_mask.sum())
)

if novel_attack_mask.any():
    novel_actual = y_attack_test.loc[novel_attack_mask]

    novel_predictions = attack_test_predictions[
        novel_attack_mask.to_numpy()
    ]

    print(
        classification_report(
            novel_actual,
            novel_predictions,
            labels=attack_classes,
            digits=4,
            zero_division=0
        )
    )

    print(
        "Novel attack-pattern Macro F1:",
        round(f1_score(
            novel_actual,
            novel_predictions,
            labels=attack_classes,
            average="macro",
            zero_division=0
        ), 4)
    )

Novel attack-pattern test records: 38280
                precision    recall  f1-score   support

      Analysis     0.1217    0.3714    0.1833       552
      Backdoor     0.1260    0.4560    0.1975       579
           DoS     0.4647    0.2839    0.3525      2085
      Exploits     0.8736    0.6816    0.7657      9330
       Fuzzers     0.8328    0.7372    0.7821      5818
       Generic     0.9977    0.9688    0.9830     17149
Reconnaissance     0.7888    0.8821    0.8328      2358
     Shellcode     0.2382    0.9288    0.3792       365
         Worms     0.3929    0.7500    0.5156        44

      accuracy                         0.8039     38280
     macro avg     0.5374    0.6733    0.5546     38280
  weighted avg     0.8667    0.8039    0.8262     38280

Novel attack-pattern Macro F1: 0.5546


In [11]:
import joblib
import numpy as np
import sklearn

multiclass_artifact = {
    "pipeline": weighted_attack_pipeline,
    "input_columns": feature_columns,
    "classes": (
        weighted_attack_pipeline
        .named_steps["classifier"]
        .classes_
        .tolist()
    ),
    "sklearn_version": sklearn.__version__,
    "model_name": "Weighted Random Forest attack classifier",
    "scope": "Known attack categories; trained on attack records only",
    "status": "Prototype; weak performance on several attack categories"
}

model_directory = project_root / "models"
model_directory.mkdir(parents=True, exist_ok=True)

multiclass_path = (
    model_directory / "attack_category_rf.joblib"
)

joblib.dump(multiclass_artifact, multiclass_path)

# Verify predictions survive saving and loading
loaded_multiclass = joblib.load(multiclass_path)

sample = X_attack_validation.iloc[:10][feature_columns]

np.testing.assert_array_equal(
    weighted_attack_pipeline.predict(sample),
    loaded_multiclass["pipeline"].predict(sample)
)

np.testing.assert_allclose(
    weighted_attack_pipeline.predict_proba(sample),
    loaded_multiclass["pipeline"].predict_proba(sample)
)

print("Saved model:", multiclass_path)
print("Multiclass reload check passed")

Saved model: D:\PROGRAMMING\NetGuard AI\netguard_ai\models\attack_category_rf.joblib
Multiclass reload check passed


In [12]:
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score
)

binary_artifact = joblib.load(
    project_root / "models/binary_rf_baseline.joblib"
)

category_artifact = joblib.load(
    project_root / "models/attack_category_rf.joblib"
)

test_df = pd.read_csv(
    project_root / "data/raw/UNSW_NB15_testing-set.csv"
)

binary_pipeline = binary_artifact["pipeline"]
category_pipeline = category_artifact["pipeline"]

X_binary_test = test_df[binary_artifact["input_columns"]]

attack_column = list(
    binary_pipeline.named_steps["classifier"].classes_
).index(1)

binary_scores = binary_pipeline.predict_proba(
    X_binary_test
)[:, attack_column]

flagged_as_attack = (
    binary_scores >= binary_artifact["threshold"]
)

# Start with Normal; assign categories only to flagged records
combined_predictions = np.full(
    len(test_df),
    "Normal",
    dtype=object
)

if flagged_as_attack.any():
    flagged_records = test_df.loc[
        flagged_as_attack,
        category_artifact["input_columns"]
    ]

    combined_predictions[flagged_as_attack] = (
        category_pipeline.predict(flagged_records)
    )

print("Total records:", len(test_df))
print("Predicted Normal:", int((~flagged_as_attack).sum()))
print("Flagged as Attack:", int(flagged_as_attack.sum()))

Total records: 82332
Predicted Normal: 27318
Flagged as Attack: 55014


In [13]:
combined_actual = test_df["attack_cat"]

combined_classes = [
    "Normal",
    *category_artifact["classes"]
]

print(
    classification_report(
        combined_actual,
        combined_predictions,
        labels=combined_classes,
        digits=4,
        zero_division=0
    )
)

print(
    "Combined accuracy:",
    round(
        accuracy_score(combined_actual, combined_predictions),
        4
    )
)

print(
    "Combined Macro F1:",
    round(
        f1_score(
            combined_actual,
            combined_predictions,
            labels=combined_classes,
            average="macro",
            zero_division=0
        ),
        4
    )
)

                precision    recall  f1-score   support

        Normal     0.9799    0.7235    0.8324     37000
      Analysis     0.0676    0.3323    0.1123       677
      Backdoor     0.0471    0.4597    0.0855       583
           DoS     0.4148    0.1458    0.2157      4089
      Exploits     0.8315    0.5701    0.6764     11132
       Fuzzers     0.2975    0.6245    0.4030      6062
       Generic     0.9974    0.9702    0.9836     18871
Reconnaissance     0.8127    0.8464    0.8292      3496
     Shellcode     0.1664    0.9312    0.2823       378
         Worms     0.3511    0.7500    0.4783        44

      accuracy                         0.7244     82332
     macro avg     0.4966    0.6354    0.4899     82332
  weighted avg     0.8603    0.7244    0.7697     82332

Combined accuracy: 0.7244
Combined Macro F1: 0.4899


In [14]:
actual_attack = test_df["label"].to_numpy() == 1

false_alarms = (
    (~actual_attack) & flagged_as_attack
).sum()

missed_attacks = (
    actual_attack & (~flagged_as_attack)
).sum()

wrong_attack_category = (
    actual_attack
    & flagged_as_attack
    & (combined_predictions != combined_actual.to_numpy())
).sum()

correct_attack_category = (
    actual_attack
    & flagged_as_attack
    & (combined_predictions == combined_actual.to_numpy())
).sum()

print("Normal traffic flagged as attack:", int(false_alarms))
print("Attacks missed as normal:", int(missed_attacks))
print("Detected attacks with wrong category:", int(wrong_attack_category))
print("Detected attacks with correct category:", int(correct_attack_category))

Normal traffic flagged as attack: 10230
Attacks missed as normal: 548
Detected attacks with wrong category: 11911
Detected attacks with correct category: 32873
